<div style="padding:26px 30px;background:linear-gradient(135deg,#0f172a,#1e293b,#0f3460);color:#f8fafc;border-radius:14px;border-left:8px solid #38bdf8;box-shadow:0 6px 20px rgba(0,0,0,0.22);">
  <h1 style="margin:0 0 10px 0;font-size:28px;font-weight:800;letter-spacing:-0.4px;color:#ffffff;">
    Squares in Squares: FTMC + Exact Staircase MILP Refill (<code style="color:#38bdf8;">fill_rect</code>) + GPU Seam-Welding (<code style="color:#4ade80;">22.311627</code>)
  </h1>
  <p style="margin:0;font-size:15.5px;line-height:1.6;color:#cbd5e1;">
    <b>Private Technical Briefing:</b> How we jumped from our <code>22.376289</code> ensemble baseline down to <b><code>22.311627</code></b> (<b>–0.06466 LB</b>), why direct full-puzzle perturbations stall on upright zero-modes, and how decoupling the <b>Tilted Core ($T$)</b> from the <b>Upright Staircase Fill ($U$)</b> via exact simplicial reduction + MILP (<code style="color:#38bdf8;">fill_rect</code>) unlocked <b>39 rectangular gadget basin jumps in 7.5 minutes</b> (including non-$45^\circ$ packings breaking below the $1/\sqrt{2} \approx 0.7071$ barrier).
  </p>
</div>


In [ ]:
import math
import shutil
import time
from pathlib import Path

import matplotlib.patches as mpatches
import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
from matplotlib.collections import PatchCollection
from matplotlib.colors import LinearSegmentedColormap
from scipy.spatial import cKDTree

plt.rcParams["figure.dpi"] = 120
plt.rcParams["savefig.dpi"] = 120
plt.rcParams["font.sans-serif"] = ["DejaVu Sans", "Arial", "Helvetica"]
TILT_CMAP = LinearSegmentedColormap.from_list(
    "tilt_cmap", ["#1d4ed8", "#0ea5e9", "#10b981", "#f59e0b", "#ef4444"]
)

DATA_DIR = Path("/kaggle/input/squares-in-squares-22-315-data")
if not DATA_DIR.exists():
    DATA_DIR = Path("/kaggle/input/datasets/cdeotte/squares-in-squares-22-315-data")
if not DATA_DIR.exists():
    DATA_DIR = Path("/data/kaggle/eda/dataset_22_315")

zdata = np.load(DATA_DIR / "gadgets_39_and_baselines.npz")
L_champ = zdata["L_champ"]
L_ens22376 = zdata["L_ens22376"]
meta_39 = zdata["meta_39"]
top8_ns = zdata["top8_ns"]

ns_all = np.arange(1, 1001)
score_ens = float(np.sum(L_ens22376 / np.sqrt(ns_all) - 1.0))
score_champ = float(np.sum(L_champ / np.sqrt(ns_all) - 1.0))
imp_count = int(np.sum(L_champ < L_ens22376 - 1e-6))

print(f"Loaded dataset from: {DATA_DIR}")
print(f"1. Starting Ensemble Baseline Score:  {score_ens:.10f}")
print(f"2. Current Champion Score:            {score_champ:.10f}")
print(f"   -> Total LB Gain:                 -{score_ens - score_champ:.6f} across {imp_count} improved puzzles!")
print(f"   -> Loaded all {len(meta_39)} Before/After FTMC-RECT gadget basin jump pairs.")


---
## 1. Why Incremental Full-Puzzle Tweaks Stall — And How `FTMC + fill_rect` Jumps Basins

When we audited why local moves on full $N \approx 500\text{--}1000$ packings started saturating around `22.316`, two structural bottlenecks stood out:
1. **Upright Flat Zero-Modes Jam Full-Puzzle Quenching:** In a $K \times K$ puzzle with $n \approx 700$ squares, roughly $500+$ squares are axis-aligned ($0^\circ$) upright blocks. Those upright blocks have over $100$ flat zero-energy sliding modes ("rattlers"). If you apply a large kick or loosen a full puzzle by $2\%$ to jump basins, the upright rows drift slightly out of lattice alignment and jam GPU L-BFGS into a high-bloat local minimum before the tilted core can reorganize.
2. **Two-Ribbon Bottleneck Masking:** In a 2-gadget anti-diagonal packing, the puzzle side length is $L = K + \max(f_1, f_2)$. Even if a radical mutation inside a full puzzle improves Ribbon 1 from $f_1 = 0.92 \to 0.88$, Ribbon 2 is still sitting at $f_2 = 0.92$, so the full puzzle's bounding box $L$ does not shrink at all and the move is rejected!

### The 3-Stage Basin-Jumping Architecture
To jump directly between distinct layout basins without upright jamming or second-ribbon masking, we decouple the problem into three stages:

* **Stage 1 — Free Tilted Monte Carlo (`FTMC`) in Isolated $(w, h, \Delta m)$ Gadget Space:**
  Instead of perturbing all squares in a full puzzle, we isolate a single rectangular gadget $(w, h, \Delta m)$ and **strip away 100% of its upright squares**, keeping only the tilted core $T$ ($\sim 15\text{--}120$ tilted squares). On $T$ alone, we apply four macro-mutations:
  1. **$\Delta m - 1 \to \Delta m$ Core Insertion:** Take the tighter tilted core $T$ from $(w, h, \Delta m - 1)$, open a slot along the band axis $\vec{u} = (w, h)/\|(w, h)\|$, and insert $+1$ new tilted square directly inside the lozenge.
  2. **Boundary-Walk Taper Reshaping:** Walk along the tilted/upright interface, converting $1\text{--}2$ adjacent upright squares into tilted squares (growing a corner bulb) while dropping $1\text{--}3$ tilted squares from the opposite taper tip so an extra staircase row of uprights can fit.
  3. **Rational-Slope Non-$45^\circ$ Re-Angling & Shear:** Rotate and shear the tilted band toward rational slopes $\theta \in \{\arctan(1/2)=26.57^\circ,\, 28.07^\circ,\, \arctan(2/3)=33.69^\circ,\, \arctan(3/4)=36.87^\circ,\, 40^\circ,\, 45^\circ\}$.
* **Stage 2 — Exact Simplicial + MILP Upright Staircase Refill (`fill_rect`):**
  Once the mutated tilted core $T_{\text{new}}$ is placed inside $[0, w + f_{\text{try}}] \times [0, h + f_{\text{try}}]$, we **do not guess upright positions**. Instead, `fill_rect(W, H, T_new)` generates:
  - All 4 wall-anchored integer lattices $\{0.5 + i,\, W - 0.5 - i\} \times \{0.5 + j,\, H - 0.5 - j\}$, **plus**
  - All **staircase upright candidates** pushed flush against the exact polygon edges of every tilted square in $T_{\text{new}}$ (up to 4 steps deep).
  - It then applies **exact simplicial corner-rule graph reduction** (resolving $>90\%$ of upright squares in $<0.5\text{ ms}$) followed by per-component **Maximum Independent Set MILP (`scipy.optimize.milp` / HiGHS)** to pack the exact maximum number of non-overlapping upright squares around $T_{\text{new}}$ in $\sim 20\text{ ms}$!
* **Stage 3 — GPU Loosen-Then-Squeeze + Seam-Welding & Void-Teleportation:**
  Because `fill_rect` hands the L40S GPU a crystal-clean seed $T_{\text{new}} \cup U_{\text{exact}}$ with initial overlap $\le 0.022$, batched CUDA L-BFGS (`lbfgs_rect`) + 6-core Sequential LP (`rect_polish`) rapidly quenches the new basin to strict feasibility ($\text{gap} > 10^{-9}$). Improved gadgets are then welded into full $K \times K$ puzzles via **4D Zero-Buffer DP** and **GPU Seam-to-Void Teleportation** (where any colliding square in the shared seam cell $[w_1, w_1+f] \times [h_2, h_2+f]$ is teleported into the largest exact-clearance pocket `exact_holes` near the lozenge tips).


In [ ]:
# Schematic Diagram of the 3-Stage Pipeline: FTMC -> fill_rect (Simplicial + MILP) -> GPU Seam-Welding & Void-Teleportation
fig, axes = plt.subplots(1, 3, figsize=(18, 5.6))

# Panel 1: Stage 1 (FTMC on Tilted Core T alone)
ax = axes[0]
ax.set_xlim(-0.3, 6.3); ax.set_ylim(-0.3, 10.3); ax.set_aspect("equal")
ax.add_patch(mpatches.Rectangle((0, 0), 5.85, 9.85, fill=True, facecolor="#f8fafc", edgecolor="#0f172a", lw=2))
ax.add_patch(mpatches.Rectangle((0, 0), 5.0, 9.0, fill=False, edgecolor="#94a3b8", lw=1.2, ls="--"))
# Draw a tilted core T with an inserted square and tapered tip
for k_i, (cx, cy, ang_d, col, lab) in enumerate([
    (0.85, 1.1, 33.7, "#f59e0b", None), (1.75, 2.3, 33.7, "#f59e0b", None),
    (2.35, 3.6, 33.7, "#10b981", "+1 Inserted\nTilted Sq"), (3.15, 4.9, 33.7, "#f59e0b", None),
    (3.95, 6.2, 33.7, "#f59e0b", None), (4.75, 7.5, 33.7, "#f59e0b", None),
    (1.65, 0.9, 33.7, "#f59e0b", None), (2.55, 2.2, 33.7, "#f59e0b", None),
    (3.40, 3.5, 33.7, "#f59e0b", None), (4.25, 4.8, 33.7, "#f59e0b", None),
]):
    a = math.radians(ang_d)
    c, s = 0.5*math.cos(a), 0.5*math.sin(a)
    pts = np.array([[cx-c+s, cy-s-c], [cx+c+s, cy+s-c], [cx+c-s, cy+s+c], [cx-c-s, cy-s+c]])
    ax.add_patch(mpatches.Polygon(pts, closed=True, facecolor=col, edgecolor="#0f172a", lw=1.2, alpha=0.88))
    if lab:
        ax.annotate(lab, xy=(cx, cy), xytext=(0.4, 5.8),
                    arrowprops=dict(arrowstyle="->", lw=1.8, color="#059669"),
                    fontsize=9.5, fontweight="bold", color="#065f46",
                    bbox=dict(boxstyle="round,pad=0.3", fc="#d1fae5", ec="#059669"))
# Ghost dropped tip square
ax.add_patch(mpatches.Rectangle((4.6, 8.3), 0.9, 0.9, fill=False, edgecolor="#ef4444", lw=1.8, ls=":"))
ax.text(3.3, 9.1, "Taper Drop (-1)\nfrees upright row", fontsize=9, fontweight="bold", color="#dc2626")
ax.annotate("", xy=(4.7, 8.2), xytext=(1.2, 1.0), arrowprops=dict(arrowstyle="<->", lw=1.8, color="#b45309", ls="--"))
ax.text(2.9, 1.2, "Re-angle & Shear T\n(e.g. 33.7°, 26.6°, 45°)", fontsize=9, fontweight="bold", color="#b45309")
ax.set_title("Stage 1: FTMC on Isolated Tilted Core T\n(Strip Uprights -> Insert / Taper / Re-Angle)", fontsize=11.5, fontweight="bold")
ax.set_xticks([]); ax.set_yticks([])

# Panel 2: Stage 2 (fill_rect Simplicial Corner-Rule + MILP Staircase Refill)
ax = axes[1]
ax.set_xlim(-0.3, 6.3); ax.set_ylim(-0.3, 10.3); ax.set_aspect("equal")
ax.add_patch(mpatches.Rectangle((0, 0), 5.85, 9.85, fill=True, facecolor="#f8fafc", edgecolor="#0f172a", lw=2))
# Corner-reduced upright squares (blue)
for ux, uy in [(0.5,9.35),(1.5,9.35),(2.5,9.35),(3.5,9.35),(0.5,8.35),(1.5,8.35),(2.5,8.35),(0.5,7.35),(1.5,7.35),(0.5,6.35),
               (5.35,0.5),(4.35,0.5),(3.35,0.5),(5.35,1.5),(4.35,1.5),(5.35,2.5),(5.35,3.5),(5.35,4.5)]:
    ax.add_patch(mpatches.Rectangle((ux-0.5, uy-0.5), 1.0, 1.0, facecolor="#93c5fd", edgecolor="#1e3a8a", lw=1.0, alpha=0.85))
# Staircase MILP kernel upright squares touching tilted core (cyan/green highlight)
for ux, uy in [(0.5, 5.15), (1.5, 6.15), (2.5, 7.15), (3.5, 8.15), (3.35, 1.65), (4.35, 2.85), (5.35, 5.65)]:
    ax.add_patch(mpatches.Rectangle((ux-0.5, uy-0.5), 1.0, 1.0, facecolor="#34d399", edgecolor="#065f46", lw=1.5, alpha=0.9))
# Tilted core T
for cx, cy, ang_d in [(0.85,1.1,33.7),(1.75,2.3,33.7),(2.35,3.6,33.7),(3.15,4.9,33.7),(3.95,6.2,33.7),(4.75,7.5,33.7),
                      (1.65,0.9,33.7),(2.55,2.2,33.7),(3.40,3.5,33.7),(4.25,4.8,33.7)]:
    a = math.radians(ang_d); c, s = 0.5*math.cos(a), 0.5*math.sin(a)
    pts = np.array([[cx-c+s, cy-s-c], [cx+c+s, cy+s-c], [cx+c-s, cy+s+c], [cx-c-s, cy-s+c]])
    ax.add_patch(mpatches.Polygon(pts, closed=True, facecolor="#f59e0b", edgecolor="#0f172a", lw=1.1, alpha=0.88))
ax.text(0.3, 9.0, "Simplicial\nCorner Rule\n(<0.5 ms)", fontsize=8.8, fontweight="bold", color="#1e3a8a")
ax.annotate("Staircase MILP\nCandidates\nTouching T", xy=(2.5, 7.15), xytext=(3.1, 8.4),
            arrowprops=dict(arrowstyle="->", lw=1.6, color="#065f46"),
            fontsize=8.8, fontweight="bold", color="#065f46",
            bbox=dict(boxstyle="round,pad=0.25", fc="#d1fae5", ec="#059669"))
ax.set_title("Stage 2: Exact Staircase MILP Refill (fill_rect)\n(Max Independent Set around T in ~20 ms)", fontsize=11.5, fontweight="bold")
ax.set_xticks([]); ax.set_yticks([])

# Panel 3: Stage 3 (2-Gadget Zero-Buffer Seam-Welding + Void-Teleportation)
ax = axes[2]
ax.set_xlim(-0.5, 10.5); ax.set_ylim(-0.5, 10.5); ax.set_aspect("equal")
ax.add_patch(mpatches.Rectangle((0, 0), 9.85, 9.85, fill=True, facecolor="#f8fafc", edgecolor="#0f172a", lw=2))
# Upright continents
ax.add_patch(mpatches.Rectangle((0, 6.2), 3.6, 3.65, facecolor="#dbeafe", edgecolor="#2563eb", lw=1.5))
ax.text(1.8, 8.0, "Upright\nContinent A", ha="center", va="center", fontsize=9.5, fontweight="bold", color="#1e40af")
ax.add_patch(mpatches.Rectangle((3.85, 0), 6.0, 6.0, facecolor="#dbeafe", edgecolor="#2563eb", lw=1.5))
ax.text(6.85, 3.0, "Upright\nContinent B", ha="center", va="center", fontsize=10, fontweight="bold", color="#1e40af")
# Gadget 1 (Vertical Ribbon) and Gadget 2 (Horizontal Ribbon)
ax.add_patch(mpatches.Rectangle((0, 0), 3.85, 6.2, facecolor="#fef3c7", edgecolor="#d97706", lw=2, alpha=0.85))
ax.text(1.9, 2.8, "Gadget 1\n(w1 x h1 + dm1)\nVertical Ribbon", ha="center", va="center", fontsize=9.2, fontweight="bold", color="#92400e")
ax.add_patch(mpatches.Rectangle((3.6, 6.0), 6.25, 3.85, facecolor="#fde68a", edgecolor="#b45309", lw=2, alpha=0.85))
ax.text(7.1, 7.9, "Gadget 2\n(w2 x h2 + dm2)\nHorizontal Ribbon", ha="center", va="center", fontsize=9.2, fontweight="bold", color="#92400e")
# Shared seam cell & Void teleportation arrow
ax.add_patch(mpatches.Rectangle((3.5, 5.9), 0.7, 0.7, facecolor="#ef4444", edgecolor="#7f1d1d", lw=2))
ax.add_patch(mpatches.Circle((1.1, 5.3), 0.42, facecolor="#4ade80", edgecolor="#166534", lw=2, ls="--"))
ax.annotate("Seam-to-Void\nTeleportation!", xy=(1.2, 5.3), xytext=(4.3, 5.1),
            arrowprops=dict(arrowstyle="->", connectionstyle="arc3,rad=-0.25", lw=2.2, color="#dc2626"),
            fontsize=9.5, fontweight="bold", color="#991b1b",
            bbox=dict(boxstyle="round,pad=0.3", fc="#fee2e2", ec="#dc2626"))
ax.set_title("Stage 3: GPU Seam-Welding & Void-Teleportation\n(Teleport Seam Collision -> Exact-Clearance Void)", fontsize=11.5, fontweight="bold")
ax.set_xticks([]); ax.set_yticks([])

plt.tight_layout()
plt.show()


---
## 2. Score Progression (`22.5919` $\to$ `22.3152`) & Per-Band Gain Breakdown

The chart below shows our progression across every stage of this campaign, along with the per-band score improvements ($\Delta \text{Score} = \sum_{n \in \text{band}} \Delta L(n)/\sqrt{n}$) from our `22.376289` ensemble baseline down to **`22.311627`**.


In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(16, 5.2))

stages = [
    ("Public SOTA\n(lightningv08)", 22.591926, "#64748b"),
    ("Our 1st #1 LB\n(4D Fault DP)", 22.481450, "#3b82f6"),
    ("Our Sequel\n(L-Cut Squeeze)", 22.399175, "#0ea5e9"),
    ("26-Notebook\nGrand Ensemble", 22.376289, "#8b5cf6"),
    ("CLIFF + Seam\nWeld + Teleport", 22.317423, "#10b981"),
    ("Graft + FTMC\n+ MILP Refill", score_champ, "#f59e0b"),
]
ax = axes[0]
names = [s[0] for s in stages]
vals = [s[1] for s in stages]
cols = [s[2] for s in stages]
bars = ax.bar(names, vals, color=cols, edgecolor="#0f172a", lw=1.3, width=0.62)
ax.set_ylim(22.25, 22.64)
ax.set_ylabel("Competition Score  Σ (L_n / √n - 1)", fontsize=11, fontweight="bold")
ax.set_title("Campaign Score Progression (22.5919 -> 22.3152)", fontsize=12.5, fontweight="bold")
ax.grid(axis="y", ls="--", alpha=0.4)
for b, v in zip(bars, vals):
    ax.text(b.get_x() + b.get_width()/2, v + 0.008, f"{v:.6f}", ha="center", va="bottom", fontsize=9.5, fontweight="bold")
ax.tick_params(axis="x", labelsize=9)

# Right panel: Cumulative & Per-Band Gain from 22.376289 -> 22.311627
ax2 = axes[1]
gains_per_n = (L_ens22376 - L_champ) / np.sqrt(ns_all)
Ks = np.floor(np.sqrt(ns_all)).astype(int)
band_ks = list(range(8, 32))
band_gains = [float(np.sum(gains_per_n[Ks == k])) for k in band_ks]
ax2.bar([f"K={k}" for k in band_ks], band_gains, color="#0ea5e9", edgecolor="#0f172a", lw=1.0)
ax2.set_ylabel("LB Score Reduction in Band K", fontsize=11, fontweight="bold")
ax2.set_title(f"Where We Gained -0.06109 LB vs. 22.376 Ensemble ({imp_count} Puzzles Improved)", fontsize=12.5, fontweight="bold")
ax2.tick_params(axis="x", rotation=45, labelsize=8.5)
ax2.grid(axis="y", ls="--", alpha=0.4)

plt.tight_layout()
plt.show()


---
## 3. Complete Before & After Gallery of All 39 `FTMC + fill_rect` Gadget Basin Jumps (7.5-Minute Run)

Below are **all 39 rectangular gadgets** $(w, h, \Delta m)$ where our 7.5-minute run of `gpu_ftmc_rect_basin_jump.py` (`FTMC` Tilted-Core Mutation + `fill_rect` Exact Simplicial/MILP Upright Staircase Refill + GPU L-BFGS + Sequential LP Polish) jumped to a tighter basin, sorted from largest bloat drop ($\Delta f = -0.105950$) to smallest:

* **Non-$45^\circ$ Sub-`0.7071` Breakthroughs:** Look closely at **#7 `(6x20+3): 0.746680 -> 0.697181` ($\Delta f = -0.049499$)** and **#8 `(7x22+4): 0.746680 -> 0.703280` ($\Delta f = -0.043400$)**. A $45^\circ$ tilted square has half-extent $1/\sqrt{2} \approx 0.707107$, which acts as a hard floor for standard $45^\circ$ lozenge crops. By re-angling the tilted band to shallower rational angles ($\sim 26^\circ\text{--}34^\circ$) and refilling the staircase uprights via `fill_rect`, both gadgets broke cleanly **below `0.7071`**!
* **Massive Narrow-Ribbon Jumps:** On narrow $w \in \{3, 4\}$ ribbons (**#1 `(3x14+4)` $-0.1060$**, **#2 `(3x14+5)` $-0.0868$**, **#3 `(3x16+5)` $-0.0850$**, **#4 `(3x15+3)` $-0.0693$**, **#5 `(4x21+3)` $-0.0659$**), inserting a tilted square into the $\Delta m - 1$ core or tapering the lozenge tip allowed `fill_rect` to lock an extra upright staircase column along the wall, collapsing the bounding box bloat by $0.05\text{--}0.10$ units!
* **High-Surplus DP Leverage Jumps:** Look at **#28 `(7x21+15): 0.956821 -> 0.947566` ($\Delta f = -0.009254$)** and **#31 `(5x14+8): 0.904622 -> 0.899231` ($\Delta f = -0.005391$)** — those single gadget jumps immediately cascaded through our 4D Zero-Buffer DP to improve full puzzles `n=503, 702, 813, 871`!

### Part A: Jumps #1 to #13 ($\Delta f$ from `-0.105950` to `-0.037941`)


In [ ]:
def half_ext(a):
    return 0.5 * (np.abs(np.cos(a)) + np.abs(np.sin(a)))

def square_corners(x, y, a):
    c, s = 0.5 * math.cos(a), 0.5 * math.sin(a)
    return np.array([
        [x - c + s, y - s - c],
        [x + c + s, y + s - c],
        [x + c - s, y + s + c],
        [x - c - s, y - s + c],
    ])

def draw_gadget_pair(ax, sq_b, sq_a, w, h, dm, fo, fn, rank_idx):
    # Plot Before (left) and After (right) side-by-side in the same axes for crisp comparison!
    he_b = half_ext(sq_b[:, 2])
    sb = sq_b.copy()
    sb[:, 0] -= (sb[:, 0] - he_b).min(); sb[:, 1] -= (sb[:, 1] - he_b).min()
    he_a = half_ext(sq_a[:, 2])
    sa = sq_a.copy()
    sa[:, 0] -= (sa[:, 0] - he_a).min(); sa[:, 1] -= (sa[:, 1] - he_a).min()

    gap_x = w + 2.2
    for shift_x, sq_cur, f_val, tag, box_col in [
        (0.0, sb, fo, f"Before\nf={fo:.4f}", "#dc2626"),
        (gap_x, sa, fn, f"After\nf={fn:.4f}", "#16a34a")
    ]:
        polys = [square_corners(x + shift_x, y, a) for x, y, a in sq_cur]
        degs = np.abs(((np.degrees(sq_cur[:, 2]) + 45.0) % 90.0) - 45.0)
        pc = PatchCollection(
            [mpatches.Polygon(p, closed=True) for p in polys],
            cmap=TILT_CMAP, edgecolor="#0f172a", linewidth=0.45, alpha=0.92
        )
        pc.set_clim(0, 45)
        pc.set_array(degs)
        ax.add_collection(pc)
        ax.add_patch(mpatches.Rectangle((shift_x, 0), w + f_val, h + f_val, fill=False, edgecolor=box_col, lw=1.6))
        ax.add_patch(mpatches.Rectangle((shift_x, 0), w, h, fill=False, edgecolor="#64748b", lw=0.8, ls=":"))
        ax.text(shift_x + 0.5*(w + f_val), -1.35, tag, ha="center", va="top", fontsize=7.8, fontweight="bold", color=box_col)

    ax.set_xlim(-0.5, gap_x + w + max(fo, fn) + 0.5)
    ax.set_ylim(-3.2, h + max(fo, fn) + 0.6)
    ax.set_aspect("equal")
    ax.axis("off")
    df = fn - fo
    ax.set_title(f"#{rank_idx}: ({int(w)}x{int(h)}+{int(dm)}) m={int(w*h+dm)}\nΔf = {df:+.5f}", fontsize=9.0, fontweight="bold", pad=3)

def plot_gadget_range(start_idx, end_idx, title_str):
    count = end_idx - start_idx
    ncols = 5 if count >= 13 else 4
    nrows = int(math.ceil(count / ncols))
    fig, axes = plt.subplots(nrows, ncols, figsize=(18, 4.4 * nrows))
    axes_flat = np.array(axes).ravel()
    for i in range(count):
        idx = start_idx + i
        w, h, dm, fo, fn, df = meta_39[idx]
        sq_b = zdata[f"before_{idx}"]
        sq_a = zdata[f"after_{idx}"]
        draw_gadget_pair(axes_flat[i], sq_b, sq_a, int(w), int(h), int(dm), fo, fn, idx + 1)
    for j in range(count, len(axes_flat)):
        axes_flat[j].axis("off")
    fig.suptitle(title_str, fontsize=14, fontweight="bold", y=1.01)
    plt.tight_layout()
    plt.show()

plot_gadget_range(0, 13, "FTMC + fill_rect Gadget Basin Jumps #1 to #13 (Largest Bloat Drops: -0.1060 to -0.0379)")


### Part B: Jumps #14 to #26 ($\Delta f$ from `-0.037676` to `-0.014105`)


In [ ]:
plot_gadget_range(13, 26, "FTMC + fill_rect Gadget Basin Jumps #14 to #26 (Bloat Drops: -0.0377 to -0.0141)")


### Part C: Jumps #27 to #39 ($\Delta f$ from `-0.013348` to `-0.000874`, Including High-Surplus DP Winners `#28 (7x21+15)` & `#31 (5x14+8)`)


In [ ]:
plot_gadget_range(26, 39, "FTMC + fill_rect Gadget Basin Jumps #27 to #39 (High-Surplus Ribbon Refinements)")


---
## 4. Full-Puzzle Before vs. After Gallery (`22.376` Ensemble vs. `22.315` Champion)

When improved rectangular gadgets are composed across the anti-diagonal fault line via **4D Zero-Buffer DP**, **GPU Seam-Welding + Void-Teleportation**, and **Clean-Compartment Ribbon Grafting**, they transform full $K \times K$ competition puzzles. Below are the top 8 full-puzzle improvements between our `22.376289` ensemble baseline and our current **`22.311627`** bank:


In [ ]:
fig, axes = plt.subplots(4, 4, figsize=(18, 17.5))
axes_flat = axes.ravel()

for idx, n_val in enumerate(top8_ns):
    n = int(n_val)
    K = int(math.isqrt(n))
    dm = n - K * K
    sq_b = zdata[f"puz_before_{n}"]
    sq_a = zdata[f"puz_after_{n}"]
    Lb = float(L_ens22376[n - 1])
    La = float(L_champ[n - 1])
    dL = La - Lb

    for sub_i, (sq_cur, L_val, tag, col) in enumerate([
        (sq_b, Lb, f"n={n} (K={K}, +{dm}) BEFORE\nL = {Lb:.6f}", "#dc2626"),
        (sq_a, La, f"n={n} (K={K}, +{dm}) AFTER\nL = {La:.6f} (dL={dL:+.5f})", "#16a34a")
    ]):
        ax = axes_flat[2 * idx + sub_i]
        he = half_ext(sq_cur[:, 2])
        s = sq_cur.copy()
        s[:, 0] -= (s[:, 0] - he).min(); s[:, 1] -= (s[:, 1] - he).min()
        polys = [square_corners(x, y, a) for x, y, a in s]
        degs = np.abs(((np.degrees(s[:, 2]) + 45.0) % 90.0) - 45.0)
        pc = PatchCollection(
            [mpatches.Polygon(p, closed=True) for p in polys],
            cmap=TILT_CMAP, edgecolor="#0f172a", linewidth=0.35, alpha=0.92
        )
        pc.set_clim(0, 45)
        pc.set_array(degs)
        ax.add_collection(pc)
        ax.add_patch(mpatches.Rectangle((0, 0), L_val, L_val, fill=False, edgecolor=col, lw=1.8))
        ax.set_xlim(-0.4, L_val + 0.4); ax.set_ylim(-0.4, L_val + 0.4)
        ax.set_aspect("equal"); ax.axis("off")
        ax.set_title(tag, fontsize=9.2, fontweight="bold", color=col)

plt.tight_layout()
plt.show()


---
## 5. Export `submission.csv` (`22.311627`) & Verify Feasibility


In [ ]:
t0 = time.perf_counter()
src_sub = DATA_DIR / "submission.csv"
dst_sub = Path("submission.csv")
if src_sub.resolve() != dst_sub.resolve():
    shutil.copyfile(src_sub, dst_sub)

df_sub = pd.read_csv(dst_sub)
bank_sub = {}
for n_val, grp in df_sub.groupby("n"):
    bank_sub[int(n_val)] = grp[["x", "y", "angle"]].to_numpy(dtype=np.float64)

def verify_packing(sq):
    he = half_ext(sq[:, 2])
    L = float(max(np.ptp(np.r_[sq[:, 0] - he, sq[:, 0] + he]), np.ptp(np.r_[sq[:, 1] - he, sq[:, 1] + he])))
    pairs = cKDTree(sq[:, :2]).query_pairs(1.42, output_type="ndarray")
    if len(pairs) == 0:
        return L, 1.0
    i, j = pairs[:, 0], pairs[:, 1]
    dx = sq[j, 0] - sq[i, 0]; dy = sq[j, 1] - sq[i, 1]
    c, s = np.cos(sq[:, 2]), np.sin(sq[:, 2])
    ci, si = c[i], s[i]; cj, sj = c[j], s[j]
    p1 = np.abs(dx * ci + dy * si); p2 = np.abs(-dx * si + dy * ci)
    p3 = np.abs(dx * cj + dy * sj); p4 = np.abs(-dx * sj + dy * cj)
    da = sq[i, 2] - sq[j, 2]
    r = 0.5 * (1.0 + np.abs(np.cos(da)) + np.abs(np.sin(da)))
    return L, float(np.min(np.maximum(np.maximum(p1, p2), np.maximum(p3, p4)) - r))

tot_score = 0.0
min_clearance = 1.0
for n in range(1, 1001):
    if n in bank_sub:
        L, mg = verify_packing(bank_sub[n])
        assert len(bank_sub[n]) == n, f"Count mismatch at n={n}"
        assert mg > 1e-9, f"Overlap detected at n={n}: min_gap={mg}"
        min_clearance = min(min_clearance, mg)
    else:
        L = float(math.ceil(math.sqrt(n)))
    tot_score += L / math.sqrt(n) - 1.0

print(f"Verified submission.csv ({len(df_sub):,} rows, {len(bank_sub)} non-trivial puzzles) in {time.perf_counter() - t0:.2f}s!")
print(f"Minimum pairwise clearance across all 1,000 puzzles: {min_clearance:.3e} (> 0, zero overlaps)")
print(f"FINAL CERTIFIED SUBMISSION SCORE: {tot_score:.10f} (Total sum L/sqrt(n): {tot_score + 1000.0:.10f})")
